# Liner system solver by SGD

Our goal is to solve $Ax = b$ by nn. 

In [1]:
# package imports
import torch
import torch.nn as nn
import torch.optim as optim

import matplotlib.pyplot as plt
import numpy as np
import time

In [2]:
# Problem 
A = torch.tensor([[3., 2.], [1., 4.]])
b = torch.tensor([5., 6.])

# Solution using PyTorch's built-in linear algebra solver
x_solution = torch.linalg.solve(A, b)
print("Solution using torch.linalg.solve:", x_solution)

Solution using torch.linalg.solve: tensor([0.8000, 1.3000])


Next, we are going to solve the above problem by SGD.
The idea is to minimize the loss function 
$$L(x) = \|Ax - b\|^2.$$

In [4]:
# define the loss function
def loss(x):
    return torch.linalg.norm(A @ x - b) # L2 norm of the residual

In [8]:
# initialize x with requires_grad=True to compute gradients
x = torch.zeros(2, requires_grad=True)
# set up the optimizer
optimizer = optim.Adam([x], lr=0.01)

print("Starting optimization...")
# optimization loop
for epoch in range(1000):
    optimizer.zero_grad()  # zero the gradients
    current_loss = loss(x)  # compute the loss
    current_loss.backward()  # compute gradients
    optimizer.step()  # update x

    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {current_loss.item()}, x: {x.detach().numpy()}")

print("Optimized solution:", x.detach().numpy())


Starting optimization...
Epoch 0, Loss: 7.8102498054504395, x: [0.01 0.01]
Epoch 100, Loss: 1.026669979095459, x: [0.96708137 1.0148454 ]
Epoch 200, Loss: 0.004102764185518026, x: [0.7998328 1.30001  ]
Epoch 300, Loss: 0.004527046345174313, x: [0.7998972 1.2989951]
Epoch 400, Loss: 0.0051878755912184715, x: [0.79898334 1.2986976 ]
Epoch 500, Loss: 0.01019215863198042, x: [0.8007537 1.3007449]
Epoch 600, Loss: 0.005072468891739845, x: [0.8009747 1.3009648]
Epoch 700, Loss: 0.006644037552177906, x: [0.7986185 1.2986293]
Epoch 800, Loss: 0.00972682610154152, x: [0.8007169 1.3007125]
Epoch 900, Loss: 0.004909941460937262, x: [0.80093956 1.3009349 ]
Optimized solution: [0.79908097 1.2990843 ]


The above result is pretty good. Next, I want to wrap up the above codes into a function of A and b with output x.

In [13]:
# define a linear system solver 
# input: A, b
# output: x such that Ax = b
def linear_system_solver(A, b, lr=0.01, epochs=1000):
    # justify the shape of A and b
    assert A.shape[0] == b.shape[0], "Number of rows in A must match the size of b"
    # initialize x with requires_grad=True to compute gradients
    x = torch.zeros(A.shape[1], requires_grad=True)
    # set up the optimizer
    optimizer = optim.Adam([x], lr=lr)

    # optimization loop
    for epoch in range(epochs):
        optimizer.zero_grad()  # zero the gradients
        current_loss = loss(x)  # compute the loss
        current_loss.backward()  # compute gradients
        optimizer.step()  # update x

    return x.detach().numpy()

In [14]:
print("Solution from linear_system_solver:", linear_system_solver(A, b))

Solution from linear_system_solver: [0.79908097 1.2990843 ]


In [18]:
# try an example with A of 2 by 3 matrix and b of size 2
A = torch.tensor([[3., 2., 1.], [1., 4., 2.]])
b = torch.tensor([5., 6.])

# print A and b
print("A:\n", A)
print("b:\n", b)

# solution using built-in solver (should raise an error because A is not square)
try:
    x_solution = torch.linalg.solve(A, b)
    print("Solution using torch.linalg.solve:", x_solution)
except RuntimeError as e:
    print("Error using torch.linalg.solve:", e)

# solution using our linear_system_solver
x = linear_system_solver(A, b)
print(f'Solution from linear_system_solver for non-square A: {x}')
print(f'checking A @ x: {A @ torch.tensor(x)}')


A:
 tensor([[3., 2., 1.],
        [1., 4., 2.]])
b:
 tensor([5., 6.])
Error using torch.linalg.solve: linalg.solve: A must be batches of square matrices, but they are 2 by 3 matrices
Solution from linear_system_solver for non-square A: [0.7983706 0.8650373 0.8650373]
checking A @ x: tensor([4.9902, 5.9886])


Remark: When $A$ is not square matrix, the built-in solver gives error message, while our solver find a solution among many solutions.

In [19]:
# change A to be a 3 by 3 matrix by adding a zero row
A = torch.tensor([[3., 2., 1.], [1., 4., 2.], [0., 0., 0.]])
b = torch.tensor([5., 6., 0.])
# solution using built-in solver (should raise an error because A is not square)
try:
    x_solution = torch.linalg.solve(A, b)
    print("Solution using torch.linalg.solve:", x_solution)
except RuntimeError as e:
    print("Error using torch.linalg.solve:", e)

# solution using our linear_system_solver
x = linear_system_solver(A, b)
print(f'Solution from linear_system_solver for non-square A: {x}')
print(f'checking A @ x: {A @ torch.tensor(x)}')


Error using torch.linalg.solve: torch.linalg.solve: The solver failed because the input matrix is singular.
Solution from linear_system_solver for non-square A: [0.7983706 0.8650373 0.8650373]
checking A @ x: tensor([4.9902, 5.9886, 0.0000])
